# Big-data gene-program recovery: dense GLLVM + varimax

**The pitch (what ZQE-GLLVM enables on large count data):** fit a **dense count-native GLLVM**
(Poisson, ZQE) with a modest `q` (here q=30) on `p` in the thousands of genes, then
**varimax-rotate** the loadings — and the rotated factors fall out as recognizable cell-type
programs, with their canonical marker genes as the top loadings. Fast, simple, scales.

Demo: 10x **pbmc3k** (~2300 cells, raw UMI counts), standard scRNA cleanup (drop mito/ribo/IEG,
depth-normalize by downsampling to a common total), top ~2000 HVGs → dense ZQE-GLLVM (q=30) →
varimax → per-factor hypergeometric enrichment vs canonical PBMC markers. Recovers **7 programs**
(B, CD14-mono, FCGR3A-mono, NK, CD8/cytotoxic, DC, platelet), one clean factor each.

> **Decision:** dense + varimax is the going-forward real-data story; the L1/lasso
> *sparse-discovery* angle is **scrapped** for the pitch — varimax recovers more programs, one
> clean factor each, cheaper (`pbmc_sparse_programs.ipynb` kept only for the record).
>
> **TODO (later):** the majority **T-cell** program isn't a distinct factor — likely absorbed into
> the **intercept/bias** (the baseline most cells share). Check by inspecting the genes the fitted
> bias is largest on."

In [1]:
import numpy as np, torch, scanpy as sc
import scipy.sparse as sp
import matplotlib.pyplot as plt
sc.settings.verbosity = 0

# --- identical preprocessing to the sparse-ZQE notebook ---
adata = sc.datasets.pbmc3k()
adata.var_names_make_unique()
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
keep = ~adata.var_names.str.match(r"^(MT-|RP[SL]|MRP[SL])")
IEG = {"JUN","JUNB","JUND","FOS","FOSB","EGR1","DUSP1","IER2","ZFP36","ATF3",
       "HSPA1A","HSPA1B","HSPB1","NR4A1","DNAJB1"}
keep &= ~adata.var_names.str.upper().isin(IEG)
adata = adata[:, keep].copy()
tot = np.asarray(adata.X.sum(1)).ravel()
floor = int(np.percentile(tot, 15))
adata = adata[tot >= floor].copy()
sc.pp.downsample_counts(adata, counts_per_cell=floor, random_state=0)
Xd = adata.X.toarray() if sp.issparse(adata.X) else np.asarray(adata.X)
mean = Xd.mean(0); var = Xd.var(0)
disp = np.divide(var, mean, out=np.zeros_like(mean), where=mean > 0)
pool = np.where(mean > 0.0125)[0]
hvg = pool[np.argsort(disp[pool])[::-1][:2000]]
adata = adata[:, np.sort(hvg)].copy()
genes = [s.upper() for s in adata.var_names]
Ycnt = adata.X.toarray() if sp.issparse(adata.X) else np.asarray(adata.X)
print(f"data {Ycnt.shape} (cells x genes)")

data (2296, 2000) (cells x genes)


## Dense GLLVM (ZQE, no L1) → varimax

In [2]:
from gllvm.gllvm_module import GLLVM
from gllvm.glms import PoissonGLM
from gllvm.autofit import ZQEAutoFitter
from gllvm.encoder import MapEncoderPoissonNewton

def varimax(Phi, gamma=1.0, maxit=200, tol=1e-7):
    """Kaiser varimax rotation of a (p x k) loading matrix."""
    p, k = Phi.shape
    R = np.eye(k); d = 0
    for _ in range(maxit):
        Lam = Phi @ R
        B = Phi.T @ (Lam**3 - (gamma / p) * Lam @ np.diag(np.diag(Lam.T @ Lam)))
        U, s, Vt = np.linalg.svd(B)
        R = U @ Vt
        d_new = s.sum()
        if d and d_new / d < 1 + tol:
            break
        d = d_new
    return Phi @ R

device = "cuda" if torch.cuda.is_available() else "cpu"
Q = 30
P = len(genes)
Y = torch.tensor(Ycnt, dtype=torch.float32)

# --- DENSE count-native GLLVM, fit by ZQE (no L1) — same model as the sparse notebook ---
torch.manual_seed(0)
g = GLLVM(latent_dim=Q, output_dim=P)
g.add_glm(PoissonGLM, idx=range(P), params={"T": torch.log1p})
torch.nn.init.normal_(g.wz, std=0.5); torch.nn.init.zeros_(g.bias)
g = g.to(device)
ftd = ZQEAutoFitter(
    g, encoder_factory=lambda g: MapEncoderPoissonNewton(g, lam=1.0, max_iter=30),
    l2=1e-3 / Y.shape[0], device=device, seed=0, verbose=True, store_wz_trace=False,
).fit(Y.to(device))

Wd = ftd.model.wz.detach().cpu().numpy()           # (genes x Q) dense loadings
L = varimax(Wd)                                     # varimax-rotated, interpretable
print(f"\ndense GLLVM fit + varimax done; loadings {L.shape}")

[warm-up] adam lr=0.1 (exit at lr≤0.002)


  ep  100  loss=+74.4793  gnorm=6.4195  lr=5.00e-02


  ep  200  loss=+34.9094  gnorm=4.9203  lr=5.00e-02


  ep  300  loss=+20.1548  gnorm=4.0140  lr=2.50e-02


  ep  400  loss=+13.7549  gnorm=3.5427  lr=1.25e-02


...

[refine] restart 1/8  change=—  |avg∇W|/|W|=0.0201  lr·|avg∇W|/|W|=5.89e-04  lr0=3.00e-01 lr_eff=2.92e-02  (tol=0.02)


[refine] restart 2/8  change=0.0457  |avg∇W|/|W|=0.0233  lr·|avg∇W|/|W|=3.41e-04  lr0=1.50e-01 lr_eff=1.46e-02  (tol=0.02)


[refine] restart 3/8  change=0.0252  |avg∇W|/|W|=0.0230  lr·|avg∇W|/|W|=1.68e-04  lr0=7.50e-02 lr_eff=7.31e-03  (tol=0.02)


[refine] restart 4/8  change=0.0120  |avg∇W|/|W|=0.0220  lr·|avg∇W|/|W|=8.05e-05  lr0=3.75e-02 lr_eff=3.65e-03  (tol=0.02)
  ✓ converged (restart change 0.0120 < tol 0.02)



dense GLLVM fit + varimax done; loadings (2000, 30)


## Same enrichment test — top-25 genes per factor vs known programs

In [3]:
from scipy.stats import hypergeom

MARKERS = {
    "T cell":        ["CD3D","CD3E","CD3G","TRAC","IL7R","LCK","CD2","CD7"],
    "CD8/cytotoxic": ["CD8A","CD8B","GZMK","GZMA","CCL5","NKG7","GZMH"],
    "NK":            ["GNLY","NKG7","KLRD1","KLRB1","KLRF1","NCAM1","PRF1","FGFBP2"],
    "B cell":        ["MS4A1","CD79A","CD79B","CD19","IGHM","IGHD","TCL1A"],
    "CD14 mono":     ["CD14","LYZ","S100A8","S100A9","S100A12","FCN1","VCAN"],
    "FCGR3A mono":   ["FCGR3A","MS4A7","CDKN1C","LST1","AIF1","COTL1"],
    "DC":            ["FCER1A","CST3","CLEC10A","HLA-DQA1"],
    "platelet":      ["PPBP","PF4","GP9","ITGA2B","NRGN","GNG11"],
}
gidx = {s: i for i, s in enumerate(genes)}
Mtot = len(genes); TOPN = 25
names = list(MARKERS)
present = {n: [gidx[x] for x in gl if x in gidx] for n, gl in MARKERS.items()}

E = np.zeros((Q, len(names)))
supports = []
for k in range(Q):
    supp = set(np.argsort(-np.abs(L[:, k]))[:TOPN])   # top-25 genes for this factor
    supports.append(supp)
    for c, n in enumerate(names):
        pres = present[n]; ov = len(supp & set(pres))
        p = hypergeom.sf(ov - 1, Mtot, len(pres), TOPN) if ov > 0 and pres else 1.0
        E[k, c] = -np.log10(max(p, 1e-300))

print("factor -> best-matching program (top-25 genes)")
hits = 0
for k in range(Q):
    c = int(E[k].argmax()); nlp = E[k, c]
    if nlp > 3:
        top = [genes[i] for i in np.argsort(-np.abs(L[:, k]))[:6]]
        print(f"  factor {k:2d}  -> {names[c]:13s} (-log10p={nlp:4.1f})   top: {', '.join(top)}")
        hits += 1
found = sorted({names[int(E[k].argmax())] for k in range(Q) if E[k].max() > 3})
print(f"\n{hits} factors enriched (p<1e-3); programs recovered: {found}")

factor -> best-matching program (top-25 genes)
  factor  5  -> FCGR3A mono   (-log10p= 7.0)   top: C1QA, FCGR3A, SIGLEC10, RP11-290F20.3, MS4A7, TPPP3
  factor 13  -> CD14 mono     (-log10p=10.9)   top: S100A9, S100A8, LGALS2, CST3, LYZ, CD14
  factor 15  -> DC            (-log10p= 5.2)   top: FCER1A, CLEC10A, UBE2Q1, LZTS2, DHX34, SMC2
  factor 17  -> NK            (-log10p= 8.9)   top: GZMB, GNLY, FGFBP2, NKG7, AKR1C3, PRF1
  factor 21  -> B cell        (-log10p= 7.7)   top: TCL1A, CD79A, VPREB3, LINC00926, FCER2, IGLL5
  factor 25  -> CD8/cytotoxic (-log10p= 8.4)   top: GZMK, CCL5, JAKMIP1, IL18RAP, LYAR, KLRG1
  factor 29  -> platelet      (-log10p= 7.7)   top: PPBP, PF4, SDPR, GNG11, CLU, NRGN

7 factors enriched (p<1e-3); programs recovered: ['B cell', 'CD14 mono', 'CD8/cytotoxic', 'DC', 'FCGR3A mono', 'NK', 'platelet']


## Enrichment map

In [ ]:
fig, ax = plt.subplots(figsize=(7, max(4, 0.3 * Q)))
im = ax.imshow(np.clip(E, 0, 30), aspect="auto", cmap="magma")
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(Q)); ax.set_yticklabels([f"f{k}" for k in range(Q)], fontsize=7)
ax.set_xlabel("known program"); ax.set_ylabel("varimax factor")
fig.colorbar(im, ax=ax, label="-log10 p (enrichment)")
ax.set_title("dense GLLVM + varimax factors vs known PBMC programs")
plt.tight_layout(); plt.show()